# 分布式系统工程问题

> **面试频率**: ⭐⭐⭐⭐⭐  
> **适用岗位**: Senior Data Engineer  
> **核心主题**: 幂等管道 | 重试去重 | 背压机制 | Exactly-once | 分布式事务

---

## 学习目标

1. 设计幂等的数据管道，避免重复处理导致的数据错误
2. 实现带指数退避和抖动的重试策略
3. 理解背压机制，防止系统因速度不匹配而崩溃
4. 理解 Exactly-once 的实现难点及实用替代方案
5. 比较分布式事务与最终一致性的适用场景

---

## 1. 幂等管道设计 (高频考点)

### 1.1 幂等性定义

**幂等性**: 执行多次与执行一次的结果相同。

数学表示: `f(f(x)) = f(x)`

```
非幂等操作 (危险):
  INSERT INTO orders VALUES (...)  --> 重复执行 -> 重复记录!
  balance += 100                   --> 重复执行 -> 余额多加!

幂等操作 (安全):
  INSERT OR REPLACE INTO orders WHERE id = ?   --> 覆盖写
  UPDATE orders SET status='paid' WHERE id=?   --> 设置绝对值
  DELETE WHERE id=? (第二次删除 = no-op)       --> 天然幂等
```

### 1.2 四种幂等模式

**模式 1: 天然幂等 (Natural Idempotency)**
- 按业务主键覆盖写入: `INSERT OR REPLACE`, `MERGE INTO`, `UPSERT`
- 适合维度表更新、配置同步

**模式 2: 幂等键 (Idempotency Key)**
- 每次操作生成唯一 UUID，服务端记录已处理的 key
- 重复请求直接返回已缓存的结果
- 适合支付、订单创建

**模式 3: 条件写 (Conditional Write)**
- `INSERT IF NOT EXISTS`
- 乐观锁: `UPDATE SET ... WHERE version = expected_version`

**模式 4: 分区级重处理 (Partition-based Reprocessing)**
- 先删除目标分区，再重新写入
- `DELETE FROM table WHERE date = '2024-01-01'; INSERT ...`
- 适合批处理 ETL（Hive、Spark 分区覆盖）

In [ ]:
# 幂等 ETL 类实现

import uuid
import hashlib
import json
from datetime import datetime
from typing import Dict, List, Optional, Any
from dataclasses import dataclass, field

@dataclass
class Record:
    """数据记录"""
    id: str
    data: Dict[str, Any]
    updated_at: datetime = field(default_factory=datetime.now)

class IdempotentETL:
    """
    幂等 ETL 管道
    
    支持多种幂等策略:
    1. upsert_by_key    - 按主键 UPSERT (最常用)
    2. idempotency_key  - 幂等键去重
    3. partition_overwrite - 分区覆盖写
    """
    
    def __init__(self):
        # 模拟目标数据库
        self._store: Dict[str, Record] = {}
        # 幂等键记录表 (实际用 Redis 或 DB)
        self._processed_keys: Dict[str, Any] = {}
        # 操作统计
        self._stats = {"inserts": 0, "updates": 0, "skipped": 0}
    
    # ===== 策略 1: UPSERT by primary key =====
    def upsert(self, records: List[Record]) -> Dict:
        """
        按主键 UPSERT: 存在则更新，不存在则插入
        天然幂等: 多次执行结果相同
        """
        ops = {"inserted": 0, "updated": 0}
        for record in records:
            if record.id in self._store:
                existing = self._store[record.id]
                # 只更新更新时间更新的记录
                if record.updated_at >= existing.updated_at:
                    self._store[record.id] = record
                    ops["updated"] += 1
            else:
                self._store[record.id] = record
                ops["inserted"] += 1
        return ops
    
    # ===== 策略 2: 幂等键 =====
    def process_with_idempotency_key(
        self,
        idempotency_key: str,
        operation: callable,
        *args,
        **kwargs
    ) -> Any:
        """
        使用幂等键执行操作
        如果该 key 已处理过，直接返回缓存结果
        """
        if idempotency_key in self._processed_keys:
            cached = self._processed_keys[idempotency_key]
            print(f"  [SKIP] 幂等键 {idempotency_key[:8]}... 已处理，返回缓存结果: {cached}")
            self._stats["skipped"] += 1
            return cached
        
        # 执行实际操作
        result = operation(*args, **kwargs)
        
        # 记录已处理
        self._processed_keys[idempotency_key] = result
        self._stats["inserts"] += 1
        print(f"  [DONE] 幂等键 {idempotency_key[:8]}... 处理完成: {result}")
        return result
    
    # ===== 策略 3: 分区覆盖写 =====
    def partition_overwrite(
        self,
        partition_key: str,
        new_records: List[Record]
    ) -> Dict:
        """
        分区覆盖写: 先删除该分区的旧数据，再写入新数据
        等价于 Spark partitionBy + SaveMode.Overwrite
        """
        # 删除旧分区数据
        old_keys = [
            k for k, v in self._store.items()
            if v.data.get("partition") == partition_key
        ]
        for k in old_keys:
            del self._store[k]
        
        # 写入新数据
        for record in new_records:
            record.data["partition"] = partition_key
            self._store[record.id] = record
        
        return {
            "partition": partition_key,
            "deleted": len(old_keys),
            "inserted": len(new_records)
        }
    
    def show_store(self):
        print(f"  存储内容 ({len(self._store)} 条):")
        for k, v in self._store.items():
            print(f"    {k}: {v.data}")

# ===== 演示 =====
etl = IdempotentETL()

print("=" * 60)
print("策略 1: UPSERT by primary key")
print("=" * 60)

records_v1 = [
    Record("user-1", {"name": "Alice", "score": 100}),
    Record("user-2", {"name": "Bob",   "score": 200}),
]

print("\n第一次写入:")
result = etl.upsert(records_v1)
print(f"  结果: {result}")
etl.show_store()

print("\n第二次写入 (完全相同的数据 - 模拟重复):")
result = etl.upsert(records_v1)
print(f"  结果: {result}")
etl.show_store()

print("\n第三次写入 (更新 user-1 的 score):")
records_v2 = [Record("user-1", {"name": "Alice", "score": 150})]
result = etl.upsert(records_v2)
print(f"  结果: {result}")
etl.show_store()

print("\n" + "=" * 60)
print("策略 2: 幂等键")
print("=" * 60)

etl2 = IdempotentETL()
payment_key = str(uuid.uuid4())

def process_payment(amount: float, user_id: str) -> dict:
    return {"status": "success", "amount": amount, "user": user_id, "txn_id": str(uuid.uuid4())[:8]}

print(f"\n支付幂等键: {payment_key[:8]}...")
print("第一次支付请求:")
etl2.process_with_idempotency_key(payment_key, process_payment, 100.0, "user-1")

print("\n第二次支付请求 (网络重试，相同幂等键):")
etl2.process_with_idempotency_key(payment_key, process_payment, 100.0, "user-1")

print("\n" + "=" * 60)
print("策略 3: 分区覆盖写")
print("=" * 60)

etl3 = IdempotentETL()
print("\n第一次写入 date=2024-01-01 分区:")
result = etl3.partition_overwrite("2024-01-01", [
    Record("r1", {"value": 10}),
    Record("r2", {"value": 20}),
])
print(f"  {result}")
etl3.show_store()

print("\n重新写入相同分区 (幂等):")
result = etl3.partition_overwrite("2024-01-01", [
    Record("r1", {"value": 10}),
    Record("r2", {"value": 20}),
])
print(f"  {result}")
etl3.show_store()

---

## 2. Retry & 重复数据处理 (高频考点)

### 2.1 重试策略对比

```
不好的重试:
  while True:
      try: call_api()
      except: time.sleep(1)  # 固定间隔 -> 所有客户端同时重试 -> 雪崩!

指数退避 (Exponential Backoff):
  第1次失败 -> 等 1s
  第2次失败 -> 等 2s
  第3次失败 -> 等 4s
  第4次失败 -> 等 8s
  ...
  
指数退避 + 抖动 (Jitter) [最佳实践]:
  第1次失败 -> 等 1s + random(0, 1s)
  第2次失败 -> 等 2s + random(0, 2s)
  ...
  防止惊群效应 (Thundering Herd)
```

### 2.2 去重策略

| 策略 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **Bloom Filter** | 概率型数据结构，空间效率高 | 极低内存 | 有误判率，不能删除 |
| **Redis SET** | 精确存储已处理 ID | 准确 | 内存有限，需设 TTL |
| **DB Unique Constraint** | 数据库唯一约束 | 最可靠 | 性能较低 |
| **Window-based Dedup** | 时间窗口内去重 | 适合流处理 | 窗口外重复无法检测 |

### 2.3 Dead Letter Queue (DLQ)

```
消息处理失败流程:

Queue --> [Consumer] --> 失败
               |         ^
               |  重试3次 |
               |  (指数退避)
               |
               v
            [DLQ] --> 告警 + 人工/自动处理
            
DLQ 的价值:
- 保存失败消息，不丢失
- 不阻塞正常消息处理
- 可以后续重放或分析失败原因
```

In [ ]:
# 指数退避 + 抖动 + DLQ 实现

import time
import random
import logging
from typing import Callable, List, Any, Optional
from dataclasses import dataclass, field
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

@dataclass
class Message:
    id: str
    payload: Any
    attempt: int = 0
    errors: List[str] = field(default_factory=list)

class RetryPolicy:
    """可配置的重试策略: 指数退避 + 抖动"""
    
    def __init__(
        self,
        max_attempts: int = 3,
        base_delay: float = 1.0,
        max_delay: float = 60.0,
        jitter: bool = True,
        backoff_multiplier: float = 2.0
    ):
        self.max_attempts = max_attempts
        self.base_delay = base_delay
        self.max_delay = max_delay
        self.jitter = jitter
        self.backoff_multiplier = backoff_multiplier
    
    def delay_for_attempt(self, attempt: int) -> float:
        """
        计算第 n 次重试的等待时间
        
        full jitter: random(0, min(cap, base * 2^attempt))
        """
        exponential = self.base_delay * (self.backoff_multiplier ** attempt)
        capped = min(self.max_delay, exponential)
        
        if self.jitter:
            return random.uniform(0, capped)
        return capped

class MessageProcessor:
    """带重试和 DLQ 的消息处理器"""
    
    def __init__(self, retry_policy: RetryPolicy):
        self.policy = retry_policy
        self.dlq: List[Message] = []         # Dead Letter Queue
        self.success_count = 0
        self.failure_count = 0
    
    def process(self, message: Message, handler: Callable) -> bool:
        """
        处理单条消息，失败时按策略重试
        超出最大重试次数后放入 DLQ
        """
        for attempt in range(self.policy.max_attempts):
            message.attempt = attempt + 1
            try:
                handler(message.payload)
                self.success_count += 1
                print(f"  [SUCCESS] msg={message.id} attempt={attempt + 1}")
                return True
            except Exception as e:
                error_msg = f"attempt {attempt + 1}: {str(e)}"
                message.errors.append(error_msg)
                
                if attempt < self.policy.max_attempts - 1:
                    delay = self.policy.delay_for_attempt(attempt)
                    print(f"  [RETRY]   msg={message.id} {error_msg} -> wait {delay:.3f}s")
                    time.sleep(delay)
                else:
                    print(f"  [FAILED]  msg={message.id} {error_msg} -> DLQ")
        
        # 放入 DLQ
        self.dlq.append(message)
        self.failure_count += 1
        return False
    
    def show_dlq(self):
        print(f"\nDLQ 内容 ({len(self.dlq)} 条):")
        for msg in self.dlq:
            print(f"  msg={msg.id} attempts={msg.attempt} errors={msg.errors}")
    
    def replay_dlq(self, handler: Callable):
        """重放 DLQ 中的消息"""
        print(f"\n重放 DLQ ({len(self.dlq)} 条)...")
        remaining = []
        for msg in self.dlq:
            msg.errors.clear()
            success = self.process(msg, handler)
            if not success:
                remaining.append(msg)
        self.dlq = remaining

# 演示
print("=" * 60)
print("重试策略: 指数退避 + Jitter")
print("=" * 60)

policy = RetryPolicy(
    max_attempts=3,
    base_delay=0.05,    # 50ms (演示用，实际生产用秒级)
    max_delay=0.5,
    jitter=True
)

# 显示退避延迟分布
print("\n退避延迟分布 (5次采样):")
for attempt in range(4):
    delays = [policy.delay_for_attempt(attempt) for _ in range(5)]
    avg = sum(delays) / len(delays)
    print(f"  第{attempt+1}次重试: delays={[f'{d:.3f}s' for d in delays]} avg={avg:.3f}s")

# 模拟消息处理
print("\n" + "=" * 60)
print("消息处理 + DLQ 演示")
print("=" * 60)

processor = MessageProcessor(policy)

fail_counter = {}

def sometimes_failing_handler(payload):
    """模拟第 1-2 次失败，第 3 次成功"""
    msg_id = payload["id"]
    fail_counter[msg_id] = fail_counter.get(msg_id, 0) + 1
    
    if payload.get("always_fail"):
        raise ValueError(f"消息 {msg_id} 永久失败")
    
    if fail_counter[msg_id] < 3:
        raise ConnectionError(f"临时网络错误 (尝试 {fail_counter[msg_id]})")
    
    print(f"    处理消息: {payload}")

messages = [
    Message("msg-001", {"id": "msg-001", "data": "正常消息"}),
    Message("msg-002", {"id": "msg-002", "always_fail": True, "data": "永久失败消息"}),
    Message("msg-003", {"id": "msg-003", "data": "临时失败消息"}),
]

for msg in messages:
    print(f"\n处理消息 {msg.id}:")
    processor.process(msg, sometimes_failing_handler)

print(f"\n处理统计: 成功={processor.success_count}, 失败={processor.failure_count}")
processor.show_dlq()

In [ ]:
# 去重策略: Bloom Filter vs Redis SET vs DB Unique Constraint 模拟

import hashlib
import math
from typing import Set

class SimpleBloomFilter:
    """
    简化版 Bloom Filter
    概率型数据结构: 可能误判 (false positive)，但不会漏判 (no false negative)
    """
    
    def __init__(self, capacity: int, error_rate: float = 0.01):
        # 计算最优位数组大小和哈希函数数量
        self.capacity = capacity
        self.error_rate = error_rate
        self.bit_size = self._optimal_bit_size(capacity, error_rate)
        self.hash_count = self._optimal_hash_count(self.bit_size, capacity)
        self._bits = bytearray(self.bit_size // 8 + 1)
        self._count = 0
        
        print(f"Bloom Filter: capacity={capacity}, error_rate={error_rate}")
        print(f"  bit_size={self.bit_size}, hash_count={self.hash_count}")
        print(f"  内存: {self.bit_size / 8 / 1024:.2f} KB")
    
    def _optimal_bit_size(self, n: int, p: float) -> int:
        return int(-n * math.log(p) / (math.log(2) ** 2))
    
    def _optimal_hash_count(self, m: int, n: int) -> int:
        return max(1, int(m / n * math.log(2)))
    
    def _get_bit_positions(self, item: str) -> list:
        positions = []
        for i in range(self.hash_count):
            h = hashlib.md5(f"{item}:{i}".encode()).hexdigest()
            pos = int(h, 16) % self.bit_size
            positions.append(pos)
        return positions
    
    def add(self, item: str):
        for pos in self._get_bit_positions(item):
            byte_idx = pos // 8
            bit_idx  = pos % 8
            self._bits[byte_idx] |= (1 << bit_idx)
        self._count += 1
    
    def might_contain(self, item: str) -> bool:
        """返回 True 表示'可能存在'（有误判率），False 表示'一定不存在'"""
        for pos in self._get_bit_positions(item):
            byte_idx = pos // 8
            bit_idx  = pos % 8
            if not (self._bits[byte_idx] & (1 << bit_idx)):
                return False
        return True

class ExactDeduplicator:
    """精确去重 (模拟 Redis SET + TTL)"""
    
    def __init__(self, max_size: int = 10000):
        self._seen: Set[str] = set()
        self.max_size = max_size
    
    def is_duplicate(self, message_id: str) -> bool:
        if message_id in self._seen:
            return True
        if len(self._seen) < self.max_size:
            self._seen.add(message_id)
        return False
    
    def memory_usage_bytes(self) -> int:
        # 粗略估算: 每个 UUID 约 64 bytes
        return len(self._seen) * 64

# 比较
print("=" * 60)
print("去重策略对比")
print("=" * 60)

N = 10000  # 处理 1 万条消息
DUPLICATES = 1000  # 其中 1000 条是重复的

# 生成测试消息
unique_ids = [f"msg-{i:06d}" for i in range(N)]
duplicate_ids = unique_ids[:DUPLICATES]  # 前 1000 条重复发送
all_messages = unique_ids + duplicate_ids
import random as _random
_random.shuffle(all_messages)

print(f"\n总消息数: {len(all_messages)}, 其中重复: {DUPLICATES}")

# 方案 1: Bloom Filter
print("\n方案 1: Bloom Filter")
bloom = SimpleBloomFilter(capacity=N, error_rate=0.01)
bloom_accepted = 0
bloom_rejected = 0
for msg_id in all_messages:
    if bloom.might_contain(msg_id):
        bloom_rejected += 1
    else:
        bloom.add(msg_id)
        bloom_accepted += 1
print(f"  接受: {bloom_accepted}, 拒绝: {bloom_rejected}")
print(f"  误判 (false positives): {bloom_rejected - DUPLICATES} 条")

# 方案 2: 精确去重
print("\n方案 2: 精确去重 (Redis SET 模拟)")
exact = ExactDeduplicator()
exact_accepted = 0
exact_rejected = 0
for msg_id in all_messages:
    if exact.is_duplicate(msg_id):
        exact_rejected += 1
    else:
        exact_accepted += 1
print(f"  接受: {exact_accepted}, 拒绝: {exact_rejected}")
print(f"  误判: 0 条 (精确)")
print(f"  内存使用: {exact.memory_usage_bytes() / 1024:.2f} KB")

print("\n结论:")
print(f"  Bloom Filter 内存: {bloom.bit_size / 8 / 1024:.2f} KB (有误判)")
print(f"  精确去重内存:       {exact.memory_usage_bytes() / 1024:.2f} KB (无误判)")
print("  高吞吐场景用 Bloom Filter，对准确性要求高用精确去重")

---

## 3. Backpressure (背压) 机制 (重要)

### 3.1 没有背压会怎样？

```
Producer (10,000 msg/s) --> Queue --> Consumer (1,000 msg/s)

不加控制:
  t=1s:  Queue 中有 9,000 条消息
  t=2s:  Queue 中有 18,000 条消息
  t=10s: Queue 中有 90,000 条消息 --> OOM!
  
  结果: 内存溢出, 进程崩溃, 消息丢失!

级联失败:
  Consumer OOM 崩溃 --> Producer 堆积更快 --> 整个系统崩溃
```

### 3.2 背压策略

| 策略 | 机制 | 适用场景 |
|------|------|----------|
| **有界队列** | Queue 满时阻塞 Producer 或抛异常 | 最常用 |
| **限速** | 令牌桶/漏桶算法限制 Producer 速率 | API 调用 |
| **负载卸载 (Shedding)** | 丢弃部分低优先级消息 | 实时系统 |
| **弹性扩容** | 动态增加 Consumer 数量 | Kubernetes HPA |

### 3.3 Kafka 背压

```
Kafka Consumer Lag 监控:
  kafka.consumer.group.lag > 阈值 --> 告警 --> 增加 Consumer 数量

Kafka Producer 背压:
  max.block.ms: Producer 等待 Buffer 可用的最大时间
  buffer.memory: Producer 内存缓冲区大小
  
  Buffer 满 --> Producer 等待 max.block.ms --> 超时抛异常
```

In [ ]:
# asyncio 有界队列实现背压

import asyncio
import random
import time
from typing import Optional

async def producer(
    queue: asyncio.Queue,
    num_items: int,
    produce_rate: float,     # items/second
    name: str = "producer"
):
    """
    生产者: 固定速率产生消息
    当队列满时会自动阻塞 (背压效果)
    """
    produced = 0
    interval = 1.0 / produce_rate
    start_time = asyncio.get_event_loop().time()
    
    for i in range(num_items):
        item = {"id": i, "data": f"message-{i}", "produced_at": asyncio.get_event_loop().time()}
        
        # put() 在队列满时自动阻塞 (背压!)
        queue_size_before = queue.qsize()
        await queue.put(item)
        produced += 1
        
        if i % 10 == 0:
            elapsed = asyncio.get_event_loop().time() - start_time
            actual_rate = produced / elapsed if elapsed > 0 else produce_rate
            print(f"  [{name}] 已产生 {produced} 条, 队列大小: {queue.qsize()}/{queue.maxsize}, "
                  f"实际速率: {actual_rate:.1f}/s")
        
        await asyncio.sleep(interval)
    
    print(f"  [{name}] 完成! 共产生 {produced} 条")

async def consumer(
    queue: asyncio.Queue,
    consume_rate: float,     # items/second
    name: str = "consumer"
):
    """
    消费者: 较慢速率消费消息
    """
    consumed = 0
    total_lag = 0.0
    interval = 1.0 / consume_rate
    
    while True:
        try:
            item = await asyncio.wait_for(queue.get(), timeout=2.0)
            
            # 模拟处理时间
            await asyncio.sleep(interval)
            
            lag = asyncio.get_event_loop().time() - item["produced_at"]
            total_lag += lag
            consumed += 1
            queue.task_done()
            
            if consumed % 10 == 0:
                avg_lag = total_lag / consumed
                print(f"  [{name}] 已消费 {consumed} 条, 平均延迟: {avg_lag:.3f}s")
        
        except asyncio.TimeoutError:
            print(f"  [{name}] 队列为空，消费者退出")
            break
    
    avg_lag = total_lag / consumed if consumed > 0 else 0
    print(f"  [{name}] 完成! 共消费 {consumed} 条, 平均延迟 {avg_lag:.3f}s")

async def run_backpressure_demo():
    print("=" * 60)
    print("背压演示: 有界队列")
    print("=" * 60)
    
    QUEUE_SIZE  = 10   # 有界队列，最多 10 条
    PRODUCE_RATE = 20  # 生产者: 20 条/秒
    CONSUME_RATE = 10  # 消费者: 10 条/秒 (较慢)
    NUM_ITEMS    = 50
    
    print(f"\n配置: 队列上限={QUEUE_SIZE}, 生产速率={PRODUCE_RATE}/s, 消费速率={CONSUME_RATE}/s")
    print(f"生产者速率 > 消费者速率 -> 队列会填满 -> 背压生效")
    print("\n开始运行...")
    
    queue = asyncio.Queue(maxsize=QUEUE_SIZE)  # 有界队列
    
    # 并发运行生产者和消费者
    await asyncio.gather(
        producer(queue, NUM_ITEMS, PRODUCE_RATE),
        consumer(queue, CONSUME_RATE),
    )

# 运行演示
await run_backpressure_demo()

---

## 4. Exactly-once 实现难点 (高频考点)

### 4.1 为什么 Exactly-once 很难？

```
At-most-once  (最多一次): 可能丢数据，不重试
At-least-once (至少一次): 可能重复，失败后重试
Exactly-once  (精确一次): 既不丢也不重复 -- 非常难!

难点:
1. 网络可能失败 -> 不知道消息是否已收到
2. 接收方可能崩溃 -> 已处理但未确认
3. 分布式系统中没有全局时钟

两将军问题 (Two Generals Problem):
  将军A 和 将军B 通过可能丢失的信使通信
  无法保证双方都确认对方已收到消息
  --> 分布式系统中 exactly-once 理论上无法完美实现
```

### 4.2 Kafka Exactly-once

```
Kafka 的实现:
1. 幂等 Producer (enable.idempotence=true)
   - Producer 有唯一 PID，消息有序号
   - Broker 检测重复消息 (相同 PID + 序号) 并丢弃

2. 事务 API (Kafka Transactions)
   producer.beginTransaction()
   producer.send(...)   # 写入目标 Topic
   producer.sendOffsetsToTransaction(...)  # 提交 Consumer Offset
   producer.commitTransaction()  # 原子提交
   
   --> Consumer 读取 + Producer 写入 是原子的

3. Consumer 端: isolation.level=read_committed
   只读取已提交事务的消息
```

### 4.3 实用替代方案: At-least-once + 幂等 Sink

```
在实际工程中，更常用的是:

  At-least-once 消费 (保证不丢)
  +
  幂等写入目标 (UPSERT/覆盖写)
  =
  效果等同于 Exactly-once

例子:
  Kafka Consumer 读消息 → 处理 → UPSERT 到 PostgreSQL (by message_id)
  即使消息被处理两次，数据库结果相同
  
  Spark Streaming → 写 Delta Lake (按分区覆盖写)
  重新处理某个时间窗口，结果幂等
```

---

## 5. 分布式事务 vs 最终一致性 (重要)

### 5.1 两阶段提交 (2PC) 问题

```
2PC 流程:
  Phase 1 (Prepare): Coordinator 发 PREPARE 给所有参与者
                     参与者执行事务但不提交，回复 YES/NO
  Phase 2 (Commit):  如果所有人说 YES -> 发 COMMIT
                     任何一个说 NO   -> 发 ABORT

问题: Coordinator 崩溃
  所有参与者处于 prepared (锁住资源) 状态
  没人知道该 COMMIT 还是 ABORT
  ---> 系统卡死! (Blocking Protocol)

实际使用: 只在单数据库内或短暂跨库时使用
         微服务架构中几乎不用
```

### 5.2 Saga 模式

```
将大事务拆分为小的本地事务，每步都有补偿事务 (Compensating Transaction)

例: 旅行预订
  T1: 预订航班  <--> C1: 取消航班
  T2: 预订酒店  <--> C2: 取消酒店
  T3: 扣款     <--> C3: 退款

成功流程: T1 -> T2 -> T3 (全部成功)
失败流程: T1 -> T2 -> T3 失败 -> C2 -> C1 (依次回滚)

两种实现:
1. Choreography (编排): 每个服务监听事件并触发下一步
   优点: 解耦  缺点: 流程分散，难追踪

2. Orchestration (协调): 中心化 Saga 协调器驱动流程
   优点: 流程清晰  缺点: 协调器是单点
```

In [ ]:
# Saga 模式状态机实现 (Orchestration 风格)

from enum import Enum, auto
from dataclasses import dataclass, field
from typing import List, Callable, Optional, Tuple
import random

class StepStatus(Enum):
    PENDING    = "PENDING"
    COMPLETED  = "COMPLETED"
    FAILED     = "FAILED"
    COMPENSATED = "COMPENSATED"

@dataclass
class SagaStep:
    name: str
    action: Callable       # 正向操作
    compensate: Callable   # 补偿操作
    status: StepStatus = StepStatus.PENDING
    result: Optional[Any] = None

class SagaOrchestrator:
    """
    Saga 编排器
    
    按顺序执行步骤，任何步骤失败时
    按相反顺序执行已完成步骤的补偿操作
    """
    
    def __init__(self, saga_id: str):
        self.saga_id = saga_id
        self.steps: List[SagaStep] = []
        self._completed_steps: List[SagaStep] = []
    
    def add_step(self, name: str, action: Callable, compensate: Callable) -> 'SagaOrchestrator':
        self.steps.append(SagaStep(name=name, action=action, compensate=compensate))
        return self  # 支持链式调用
    
    def execute(self, context: dict) -> Tuple[bool, dict]:
        """
        执行 Saga
        
        Returns:
            (success, context) - success=False 时已执行补偿
        """
        print(f"\n[Saga {self.saga_id}] 开始执行")
        
        for step in self.steps:
            print(f"\n  步骤: {step.name}")
            try:
                result = step.action(context)
                step.status = StepStatus.COMPLETED
                step.result = result
                self._completed_steps.append(step)
                print(f"    [OK] 完成: {result}")
            except Exception as e:
                step.status = StepStatus.FAILED
                print(f"    [FAIL] 失败: {e}")
                
                # 触发补偿
                print(f"\n  开始补偿 ({len(self._completed_steps)} 个步骤需要回滚)...")
                self._compensate(context)
                return False, context
        
        print(f"\n[Saga {self.saga_id}] 所有步骤完成!")
        return True, context
    
    def _compensate(self, context: dict):
        """按相反顺序执行补偿"""
        for step in reversed(self._completed_steps):
            print(f"  补偿步骤: {step.name}")
            try:
                step.compensate(context)
                step.status = StepStatus.COMPENSATED
                print(f"    [OK] 补偿完成")
            except Exception as e:
                print(f"    [WARN] 补偿失败 (需要人工干预): {e}")

# ===== 旅行预订 Saga 演示 =====

# 模拟外部服务
class TravelBookingService:
    booked_flights: list = []
    booked_hotels: list = []
    payments: list = []
    fail_at: Optional[str] = None  # 可设置在哪个步骤失败

svc = TravelBookingService()

def book_flight(ctx: dict) -> str:
    if svc.fail_at == "flight":
        raise Exception("航班已满")
    booking_id = f"FL-{random.randint(1000, 9999)}"
    svc.booked_flights.append(booking_id)
    ctx["flight_booking"] = booking_id
    return f"航班预订成功: {booking_id}"

def cancel_flight(ctx: dict):
    if booking_id := ctx.get("flight_booking"):
        svc.booked_flights.remove(booking_id)
        print(f"      取消航班: {booking_id}")

def book_hotel(ctx: dict) -> str:
    if svc.fail_at == "hotel":
        raise Exception("酒店已满")
    booking_id = f"HT-{random.randint(1000, 9999)}"
    svc.booked_hotels.append(booking_id)
    ctx["hotel_booking"] = booking_id
    return f"酒店预订成功: {booking_id}"

def cancel_hotel(ctx: dict):
    if booking_id := ctx.get("hotel_booking"):
        svc.booked_hotels.remove(booking_id)
        print(f"      取消酒店: {booking_id}")

def process_payment(ctx: dict) -> str:
    if svc.fail_at == "payment":
        raise Exception("支付失败: 余额不足")
    txn_id = f"TXN-{random.randint(10000, 99999)}"
    svc.payments.append(txn_id)
    ctx["payment"] = txn_id
    return f"支付成功: {txn_id}, 金额: ¥{ctx.get('amount', 0)}"

def refund_payment(ctx: dict):
    if txn_id := ctx.get("payment"):
        svc.payments.remove(txn_id)
        print(f"      退款: {txn_id}")

def build_travel_saga(saga_id: str) -> SagaOrchestrator:
    saga = SagaOrchestrator(saga_id)
    saga.add_step("预订航班", book_flight,    cancel_flight)
    saga.add_step("预订酒店", book_hotel,     cancel_hotel)
    saga.add_step("处理支付", process_payment, refund_payment)
    return saga

# 场景 1: 全部成功
print("=" * 60)
print("场景 1: 全部步骤成功")
print("=" * 60)
svc.fail_at = None
svc.booked_flights.clear(); svc.booked_hotels.clear(); svc.payments.clear()

saga1 = build_travel_saga("TRV-001")
success, ctx = saga1.execute({"amount": 5000})
print(f"\n结果: {'成功' if success else '失败'}")
print(f"已预订航班: {svc.booked_flights}")
print(f"已预订酒店: {svc.booked_hotels}")
print(f"已处理支付: {svc.payments}")

# 场景 2: 支付失败 → 补偿回滚
print("\n" + "=" * 60)
print("场景 2: 支付失败 → 补偿回滚")
print("=" * 60)
svc.fail_at = "payment"
svc.booked_flights.clear(); svc.booked_hotels.clear(); svc.payments.clear()

saga2 = build_travel_saga("TRV-002")
success, ctx = saga2.execute({"amount": 5000})
print(f"\n结果: {'成功' if success else '失败 (已补偿)'}")
print(f"已预订航班 (应为空): {svc.booked_flights}")
print(f"已预订酒店 (应为空): {svc.booked_hotels}")

---

## 复习要点

### 幂等管道
- 幂等 = 执行多次和一次结果相同: `f(f(x)) = f(x)`
- 四种模式: 天然幂等(UPSERT)、幂等键(UUID)、条件写、分区覆盖
- Spark 写 Delta/Hive 时用分区覆盖是最常见的幂等模式

### Retry & 去重
- 指数退避 + Full Jitter 防止惊群效应
- DLQ 保存失败消息，不阻塞主流程
- Bloom Filter 适合高吞吐但允许误判；精确去重适合严格场景

### Backpressure
- 有界队列是最简单有效的背压机制
- 监控 Consumer Lag 是发现背压问题的关键指标
- 负载卸载 (Shedding) 适合实时系统，牺牲部分数据保全系统

### Exactly-once
- 理论上很难，实践中用 At-least-once + 幂等 Sink 代替
- Kafka 通过幂等 Producer + 事务 API 实现
- Spark 通过 Checkpoint + 幂等写入实现

### 分布式事务
- 2PC: 阻塞协议，Coordinator 崩溃会卡死，微服务中不推荐
- Saga: 拆分为本地事务 + 补偿，是微服务的主流选择
- Choreography 解耦但难追踪；Orchestration 清晰但有单点

---

## 练习

### 练习 1: 幂等设计

设计一个**广告点击事件 ETL**的幂等管道：
- 来源: Kafka Topic `ad_clicks`
- 目标: PostgreSQL 表 `click_aggregates`（按广告 ID 和小时聚合点击数）
- 要求: 即使 Kafka Consumer 重复消费同一批消息，结果必须正确

请说明使用哪种幂等策略，并写出核心 SQL/代码。

### 练习 2: 重试策略实现

扩展 `RetryPolicy` 类，添加：
1. **可重试异常过滤**: 只有 `ConnectionError`, `TimeoutError` 才重试，`ValueError` 直接放 DLQ
2. **重试预算 (Retry Budget)**: 全局限制每秒最多重试 N 次，超出则直接 DLQ（防止重试风暴）
3. **回调钩子**: `on_retry`, `on_success`, `on_dlq` 回调函数

In [ ]:
# 练习 2: 增强版 RetryPolicy
# 在此基础上扩展

from typing import Type, Tuple

class EnhancedRetryPolicy:
    def __init__(
        self,
        max_attempts: int = 3,
        base_delay: float = 1.0,
        retryable_exceptions: Tuple[Type[Exception], ...] = (Exception,),
        retry_budget_per_second: int = 100,
        on_retry=None,
        on_success=None,
        on_dlq=None
    ):
        # TODO: 实现
        pass
    
    def call(self, func: Callable, *args, **kwargs):
        # TODO: 实现带预算和过滤的重试
        pass

# 测试
policy = EnhancedRetryPolicy(
    max_attempts=3,
    retryable_exceptions=(ConnectionError, TimeoutError),
    retry_budget_per_second=5,
    on_retry=lambda msg, attempt, err: print(f"  [RETRY] {msg} attempt={attempt}"),
    on_dlq=lambda msg, err: print(f"  [DLQ]   {msg} -> {err}"),
)

# 应该重试 ConnectionError
# 应该不重试 ValueError
# 超过重试预算时直接 DLQ

### 练习 3: 背压监控

设计一个 Kafka Consumer Lag 监控系统：
1. 如何计算 Consumer Lag？（Lag = Latest Offset - Consumer Offset）
2. Lag 超过 10,000 时应触发什么告警？
3. Lag 持续增长（消费速度 < 生产速度）时如何自动扩容？
4. 用 Python 实现一个简单的 Lag 模拟器和阈值检测器

### 练习 4: Exactly-once 场景分析

你有一个 Spark Streaming 作业：从 Kafka 读取用户行为事件，实时更新 Redis 中的用户积分。

1. 这个场景能实现真正的 Exactly-once 吗？为什么？
2. 如果 Spark 作业崩溃后重启，如何防止用户积分被重复累加？
3. 如果改用 `SET score = base_score + delta WHERE version = expected_version`，这是什么模式？

### 练习 5: Saga 补偿设计

设计一个**电商订单创建** Saga，步骤为：
1. 锁定库存
2. 创建订单
3. 扣减积分
4. 支付扣款
5. 发货通知

- 为每个步骤设计补偿操作
- 如果第 4 步（支付）失败，第 3 步（积分已扣）的补偿逻辑是什么？
- 如果补偿操作本身失败（如退款接口超时），如何处理？